In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 0.4 Linear Systems and Matrix Factorizations

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume 0 — Mathematical & Computational Foundations",
    number="0.4",
    title="Linear Systems and Matrix Factorizations",
    blurb="Solving Ax=b the way computers actually do it: conditioning, LU, "
    "QR, and Cholesky — why one factors a matrix instead of inverting "
    "it, and how a deceptively simple matrix can destroy the answer.",
    difficulty="intermediate",
    estimate="90–120 min",
)

## Notebook overview

Solving $A\mathbf x=\mathbf b$ is the most-run computation in physics: it sits
under every equilibrium, every normal-mode problem, every fit. We learned to
solve it by Gaussian elimination, and perhaps to write $\mathbf x = A^{-1}\mathbf
b$. A computer does **neither** of those literally. It **factors** $A$ into
pieces that are trivial to solve (triangular or orthogonal) and it **never
forms the inverse**, because doing so is slower *and* less accurate. This
notebook is about how that works and, just as importantly, when it **fails**:
some matrices are so **ill-conditioned** that no algorithm, however perfect, can
return a correct answer.

We will build back-substitution by hand, then meet the three factorizations a
numerical library actually uses: **LU** (the general workhorse), **QR**
(orthogonal, stable, the basis of least squares), and **Cholesky** (for
symmetric positive-definite matrices), each on an explicit, written-out matrix.
The centrepiece is the **Hilbert matrix**, a one-line definition whose condition
number explodes so fast that by $12\times12$ the solution is *destroyed* despite
a microscopic residual: the linear-algebra face of the round-off limits of
[§0.1](floating-point.ipynb) ($\text{achievable accuracy}\approx\varepsilon\,\kappa(A)$).

This is the engine under much of what follows: the eigenproblems of
[§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
[§2.6](../02-classical-mechanics/rigid-body.ipynb), and
[§2.7](../02-classical-mechanics/small-oscillations.ipynb) and the
least-squares fitting of [§0.8](fitting-least-squares.ipynb) all rest on these
factorizations. It is the first of two linear-algebra notebooks:
[§0.5](eigenvalues-svd.ipynb) takes up eigenvalues,
diagonalization, and the SVD. There are no animations here: factorization is
static algebra, and the one geometric motion worth animating (how a matrix maps
the unit circle to an ellipse) belongs with the SVD in
[§0.5](eigenvalues-svd.ipynb).

> **How to read the checks.** Each exercise ends with a `validate` call against
> an independent fact: a reconstruction $PA=LU$, an orthogonality $Q^\top Q=I$,
> a known solution. A ✓ is strong evidence; a ✗ is a prompt to *locate the
> discrepancy*, not a verdict.

> **Scope.** A working review, not a numerical-linear-algebra course. The
> standard references are Trefethen & Bau, *Numerical Linear Algebra*
> {cite}`trefethen1997`, and Golub & Van Loan, *Matrix Computations*
> {cite}`golubvanloan`; round-off limits trace back to
> [§0.1](floating-point.ipynb).

## Theory in brief

### Solve by factoring, not by inverting

To solve $A\mathbf x=\mathbf b$ a computer does not form the inverse, even though
pencil-and-paper linear algebra writes the answer as $\mathbf x=A^{-1}\mathbf b$.
Instead we factor $A$ into triangular or orthogonal pieces, on which the system
becomes trivial. A **triangular** system, in particular, is solved directly by
substitution in $O(n^2)$ operations: for an upper-triangular $U$,
back-substitution sweeps from the last row up, $x_i = \big(b_i - \sum_{j>i}
U_{ij}x_j\big)/U_{ii}$.

Why avoid the inverse? There are two decisive reasons. First, forming $A^{-1}$ is
not one solve but $n$ of them: the $j$-th column of the inverse is the solution of
$A\mathbf x_j=\mathbf e_j$ for the $j$-th unit vector $\mathbf e_j$, so assembling
the whole inverse costs roughly three times a single factor-and-solve, and one
must then *still* multiply $A^{-1}$ by $\mathbf b$. That surplus arithmetic
accumulates surplus rounding, leaving the inverse route both slower and less
accurate. Second, the inverse destroys structure: a sparse or banded $A$ (the
tridiagonal stiffness matrices of [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb) are typical) has a generally *dense*
inverse, throwing away exactly the memory and speed the structure was buying,
whereas a factorization keeps the sparsity. One almost never wants $A^{-1}$ itself;
one wants its action on one particular $\mathbf b$, and `solve` delivers precisely
that. Exercise 7 makes the cost-and-accuracy gap quantitative.

### Conditioning: the problem, not the method

How accurately *can* $A\mathbf x=\mathbf b$ be solved? That is set by the
**condition number**

```{math}
:label: eq-condition
\kappa(A) = \lVert A\rVert\,\lVert A^{-1}\rVert = \frac{\sigma_{\max}}{\sigma_{\min}},
\qquad
\frac{\lVert \Delta\mathbf x\rVert}{\lVert \mathbf x\rVert}
\;\lesssim\; \kappa(A)\,\frac{\lVert \Delta\mathbf b\rVert}{\lVert \mathbf b\rVert},
```

the ratio of largest to smallest singular value. It measures how much a relative
perturbation of the input is amplified in the output (Trefethen & Bau,
*Numerical Linear Algebra*, Lecture 12, derives the bound). Since the input is
already stored with relative error $\sim\varepsilon$
([§0.1](floating-point.ipynb)), the best achievable
accuracy is $\sim\varepsilon\,\kappa(A)$: a large $\kappa$ means even a *perfect*
algorithm loses digits. Conditioning is a property of the **problem**, not the
method.

### LU, QR, Cholesky

We now introduce **LU factorization with partial pivoting**, the general-purpose
workhorse here, and also exactly what `scipy.linalg.solve` runs under the hood.
Writing

```{math}
:label: eq-lu
PA = LU,
```

whereby $P$ is a permutation matrix (row pivoting, for numerical stability), $L$
is unit-lower-triangular, and $U$ upper-triangular, we then solve $L\mathbf y =
P\mathbf b$ and $U\mathbf x = \mathbf y$ by substitution. When orthogonality is
what we want instead, **QR factorization**

```{math}
:label: eq-qr
A = QR, \qquad Q^\top Q = I,\ R\ \text{upper-triangular},
```

writes $A$ via an orthogonal $Q$; it is numerically stable and the basis of
least squares (forward link [§0.8](fitting-least-squares.ipynb)) and the QR
eigenvalue algorithm (forward link [§0.5](eigenvalues-svd.ipynb)).
**Cholesky factorization**, for a symmetric positive-definite (SPD) $A$,

```{math}
:label: eq-chol
A = LL^\top,
```

is about twice as cheap as LU and is the natural tool for the mass/stiffness
matrices of [§2.7](../02-classical-mechanics/small-oscillations.ipynb),
covariance matrices, and any energy quadratic form.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import lu, qr, cholesky, solve, solve_triangular, hilbert

from ecp import validate

np.set_printoptions(precision=4, suppress=True)


def back_substitution(U, b):
    """Solve an upper-triangular system by back-substitution.

    The O(n^2) final step of every LU/QR solve, working from the last unknown upward.

    Parameters
    ----------
    U : numpy.ndarray
        Upper-triangular matrix.
    b : numpy.ndarray
        Right-hand side.

    Returns
    -------
    numpy.ndarray
        The solution x.
    """
    U, b = np.asarray(U, float), np.asarray(b, float)
    n = len(b)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (b[i] - U[i, i + 1 :] @ x[i + 1 :]) / U[i, i]
    return x

## Exercise 1 — Triangular systems and back-substitution

A triangular system is the easy case every
factorization reduces to. For an upper-triangular $U$, the last equation involves
only $x_n$, the second-last only $x_{n-1}, x_n$, and so on, so the unknowns follow from the
bottom up, each in one step. The cost is $O(n^2)$ (one division and a dot
product per row), and the result is exact up to rounding.

$$
U = \begin{bmatrix} 2 & 1 & 1 \\ 0 & 3 & 1 \\ 0 & 0 & 4 \end{bmatrix},
\qquad
\mathbf b = \begin{bmatrix} 5 \\ 6 \\ 8 \end{bmatrix}.
$$

1. Implement back-substitution (the `back_substitution` helper above) and solve
   this explicit system.
2. Confirm $U\mathbf x = \mathbf b$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    U1 @ x1, b1, "back-substitution solves the triangular system Ux=b", atol=1e-12
)

## Exercise 2 — LU factorization with partial pivoting

A general matrix is solved by reducing it to
triangular pieces: LU with partial pivoting, $PA=LU$ ({eq}`eq-lu`), where row
swaps recorded in $P$ keep the elimination numerically stable. Once factored, two
triangular solves ($L\mathbf y=P\mathbf b$, then $U\mathbf x=\mathbf y$) finish
the job, and the *same* factorization can be reused for many right-hand sides.

**This exercise.** Factor the explicit matrix

$$
A = \begin{bmatrix} 2 & 1 & 1 \\ 4 & 3 & 3 \\ 8 & 7 & 9 \end{bmatrix}
$$

with `scipy.linalg.lu`:

1. Factor $A$ and confirm the product of the returned factors reconstructs it.
2. Solve $A\mathbf x = \begin{bmatrix}1\\2\\3\end{bmatrix}$ with
   `scipy.linalg.solve` (LU under the hood).
3. Check the residual $\lVert A\mathbf x-\mathbf b\rVert$ sits at the rounding
   floor.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(P @ L @ Up, A2, "PA=LU reconstructs A", atol=1e-12)
validate.close(A2 @ x2, b2, "the LU solve satisfies A x = b", atol=1e-12)

## Exercise 3 — Conditioning: the Hilbert matrix catastrophe

This is the centrepiece. The **Hilbert matrix** of
size $n$ is defined entrywise by

$$
H_{ij} = \frac{1}{i+j-1}, \qquad i,j = 1,\dots,n
\quad\Longrightarrow\quad
H = \begin{bmatrix} 1 & \tfrac12 & \tfrac13 & \cdots \\
    \tfrac12 & \tfrac13 & \tfrac14 & \cdots \\
    \tfrac13 & \tfrac14 & \tfrac15 & \cdots \\
    \vdots & & & \ddots \end{bmatrix}.
$$

It looks utterly benign (small rational entries, symmetric, positive-definite),
yet its condition number $\kappa(H_n)$ of {eq}`eq-condition` grows roughly like
$e^{3.5n}$, so the achievable accuracy $\varepsilon\,\kappa$ collapses fast.

1. For $n=4,8,12$ form the Hilbert matrix $H_n$ (`scipy.linalg.hilbert`), take
   the **known exact solution** $\mathbf x=(1,1,\dots,1)$, and build
   $\mathbf b = H_n\mathbf x$.
2. Solve $H_n\hat{\mathbf x}=\mathbf b$ and compare $\hat{\mathbf x}$ to
   $\mathbf x$ alongside $\kappa(H_n)$ (`numpy.linalg.cond`).
3. Show that at $n=12$ the recovered solution is wildly wrong despite the exact
   arithmetic being flawless — $\kappa(H_{12})>10^{15}$ has consumed every digit.
   {numref}`fig-linsys-hilbert-cond` plots $\kappa(H_n)$ against $n$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    cond_H12 > 1e15 and recovered_error > 0.1,
    "the Hilbert matrix is so ill-conditioned that the n=12 solution is destroyed",
    f"κ(H₁₂) = {cond_H12:.2e}, ‖x̂ − x‖∞ = {recovered_error:.2f}",
)

## Exercise 4 — Residual versus error (the conditioning lesson, sharpened)

A natural instinct is to trust a solution whose
**residual** $\mathbf r = H\hat{\mathbf x}-\mathbf b$ is tiny. For an
ill-conditioned matrix that instinct is wrong. A backward-stable solver always
returns a small residual, but {eq}`eq-condition` relates the *error* to the
residual through $\kappa$: $\lVert\hat{\mathbf x}-\mathbf x\rVert/\lVert\mathbf
x\rVert \lesssim \kappa(H)\,\lVert\mathbf r\rVert/\lVert\mathbf b\rVert$. A small
residual can therefore hide a huge error.

1. For the same $n=12$ Hilbert system (exact $\mathbf x=(1,\dots,1)$,
   $\mathbf b=H_{12}\mathbf x$), compute the residual
   $\lVert H_{12}\hat{\mathbf x}-\mathbf b\rVert$ and the error
   $\lVert\hat{\mathbf x}-\mathbf x\rVert$ (`numpy.linalg.norm`).
2. Confirm the residual sits at the rounding floor while the error is $O(1)$ —
   and that $\kappa\cdot\lVert r\rVert/\lVert b\rVert$ correctly bounds it.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    residual12 < 1e-8 and error12 > 0.1,
    "a tiny residual can hide a huge error when κ is large",
    f"residual {residual12:.2e}, error {error12:.2f}",
)

## Exercise 5 — QR factorization

Not every matrix is square, and not every problem
wants LU. The **QR factorization** $A=QR$ ({eq}`eq-qr`) writes any
$m\times n$ matrix ($m\ge n$) as an orthogonal $Q$ (columns orthonormal,
$Q^\top Q=I$) times an upper-triangular $R$. Because orthogonal maps preserve
lengths, QR is exceptionally stable, and it is the foundation of least-squares
fitting (forward link [§0.8](fitting-least-squares.ipynb)) and the QR
eigenvalue algorithm (forward link [§0.5](eigenvalues-svd.ipynb)).

**This exercise.** Factor the explicit $3\times2$ matrix

$$
A = \begin{bmatrix} 1 & 2 \\ 3 & 4 \\ 5 & 6 \end{bmatrix}
$$

with `scipy.linalg.qr`:

1. Factor $A=QR$ in economy mode.
2. Confirm $QR$ reconstructs $A$.
3. Verify the columns of $Q$ are orthonormal ($Q^\top Q = I_2$).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(Q @ R, A5, "A=QR reconstructs A", atol=1e-12)
validate.close(Q.T @ Q, np.eye(2), "Q has orthonormal columns (QᵀQ=I)", atol=1e-12)

## Exercise 6 — Cholesky for symmetric positive-definite matrices

When $A$ is **symmetric positive-definite** (SPD),
symmetric with all eigenvalues positive, equivalently $\mathbf z^\top A\mathbf
z>0$ for all $\mathbf z\ne\mathbf 0$, it has a **Cholesky factorization**
$A=LL^\top$ ({eq}`eq-chol`) with $L$ lower-triangular. It is about twice as cheap
as LU and never needs pivoting. SPD matrices are everywhere in physics: the
mass and stiffness matrices of
[§2.7](../02-classical-mechanics/small-oscillations.ipynb), covariance
matrices, and any energy quadratic form.

**This exercise.** Take the explicit matrix

$$
A = \begin{bmatrix} 4 & 2 & 2 \\ 2 & 5 & 3 \\ 2 & 3 & 6 \end{bmatrix},
$$

and:

1. Confirm it is SPD by showing all its eigenvalues are positive
   (`numpy.linalg.eigvalsh`).
2. Factor it as $A=LL^\top$ with `scipy.linalg.cholesky`.
3. Check the reconstruction $LL^\top = A$.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    np.all(eigs > 0),
    "the matrix is symmetric positive-definite (all eigenvalues > 0)",
    f"min eigenvalue = {eigs.min():.3f}",
)
validate.close(Lc @ Lc.T, A6, "A=LLᵀ reconstructs the SPD matrix", atol=1e-12)

## Exercise 7 — Never invert: solve via factorization vs. forming A⁻¹

Writing $\mathbf x = A^{-1}\mathbf b$ is fine on
paper, but `inv(A) @ b` is the wrong way to compute it: forming $A^{-1}$ does
roughly three times the arithmetic of a factor-and-solve, and (because it solves
$n$ extra systems to build the inverse columns) it accumulates *more* rounding. On
an ill-conditioned matrix the gap is plainly visible.

1. For the Hilbert system $H_{10}$ with known exact solution
   $\mathbf x=(1,\dots,1)$ and $\mathbf b=H_{10}\mathbf x$, compare the error of
   `scipy.linalg.solve(H, b)` against `numpy.linalg.inv(H) @ b`.
2. Show the same comparison across Hilbert sizes $n=2,\dots,13$
   ({numref}`fig-linsys-solve-vs-inv`).
3. Confirm solving is at least as accurate as inverting.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    err_solve <= err_inv,
    "solving via factorization is at least as accurate as forming the inverse",
    f"solve {err_solve:.2e} ≤ inv {err_inv:.2e}",
)

## Exercise 8 — Choosing the right factorization (synthesis)

A short decision guide, each branch justified by
what we saw:

- **general $A$ → LU** ({eq}`eq-lu`): the default; what `scipy.linalg.solve` runs.
- **least squares / orthogonality wanted → QR** ({eq}`eq-qr`): stable, the basis
  of fitting ([§0.8](fitting-least-squares.ipynb)).
- **symmetric positive-definite → Cholesky** ({eq}`eq-chol`): about twice as fast,
  no pivoting (the mass/stiffness matrices of
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb)).

And always: **factor and solve, never `inv(A)@b`** (Exercise 7).

**This exercise.** For the well-conditioned explicit system

$$
A = \begin{bmatrix} 3 & 1 \\ 1 & 2 \end{bmatrix},
\qquad \mathbf b = \begin{bmatrix} 9 \\ 8 \end{bmatrix},
$$

1. Solve it by LU, via `scipy.linalg.solve`.
2. Solve it independently by QR: form $Q^\top\mathbf b$, then back-substitute
   $R\mathbf x=Q^\top\mathbf b$ with `scipy.linalg.solve_triangular`.
3. Confirm the two solutions agree. (The exact solution is $\mathbf x=(2,3)$.)

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.close(
    x_lu,
    x_qr,
    "LU and QR give the same solution to the well-conditioned system",
    rtol=1e-10,
)

## Notebook summary

- Triangular solves and **LU with partial pivoting** ($PA=LU$, the solve satisfying $A\mathbf x
  =\mathbf b$); the Hilbert matrix as a conditioning catastrophe, and residual versus error.
- **QR** ($Q^\top Q=I$) and **Cholesky** ($A=LL^\top$ for symmetric positive-definite
  matrices); and why one never forms $A^{-1}$ but solves via a factorization instead.

## Outlook

- **Sparse and banded systems.** Most physics matrices are mostly zeros;
  exploiting structure (a tridiagonal solver) is what makes the 1-D Schrödinger
  equation of [§6.10](../06-quantum-mechanics/schrodinger-on-a-computer.ipynb) tractable: $O(n)$ instead of $O(n^3)$.
- **Iterative solvers.** For huge SPD systems, conjugate gradient never forms a
  factorization at all, converging in far fewer steps than the dimension.
- **The SVD.** The ultimate "what is this matrix doing" tool, and the honest way
  to define $\kappa=\sigma_{\max}/\sigma_{\min}$, is taken up in
  [§0.5](eigenvalues-svd.ipynb).
- **Forward links.** These factorizations underlie every eigen/normal-mode
  computation ([§1.5](../01-elementary-mechanics/coupled-oscillators.ipynb),
  [§2.6](../02-classical-mechanics/rigid-body.ipynb),
  [§2.7](../02-classical-mechanics/small-oscillations.ipynb)) and the
  least-squares fit of [§0.8](fitting-least-squares.ipynb).

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()